LOAD LIBRARIES

In [6]:
import oracle_repo as orep
import importlib
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer, util

/Users/Lukas_1/anaconda3/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


CREATE A STORAGE OBJECT FOR MODEL OUTPUT and VECTOR OUTPUT THAT REMAINS VALID OVER ITERATIONS

In [4]:
response_outputs_backup = []
training_data = []

RESET THE RESPONSE OBJECT

In [5]:
response_outputs = []

PRECONFIGURE THE PARAMETERS

In [7]:
# Define the problem type
# Example usage with user input
problem_types = ["deterministic", "predictive", "code_based", "philosophical"]

print("Choose problem type:")
for p_type in problem_types:
    print(f"- {p_type}")

# Get user input for problem type
problem_type = input("Enter the problem type: ").strip().lower()

if problem_type not in problem_types:
    raise ValueError("Invalid problem_type provided")

Choose problem type:
- deterministic
- predictive
- code_based
- philosophical


In [8]:
problem_scope = ["open", "closed"]

print("Choose problem scope:")
for p_type in problem_scope:
    print(f"- {p_type}")

# Get user input for problem type
problem_scope = input("Enter the problem type (open or closed problem?): ").strip().lower()

if problem_scope not in problem_scope:
    raise ValueError("Invalid problem_type provided")


Choose problem scope:
- open
- closed


In [9]:
#Initialize parameters
adaptation_rate = 0 #initialize adaptation rate.
messages = [] #cache to store list of best_responses for each request iteration t.
best_response_backup = []
num_steps_proposed = 0

#fill first iteration of feature vector with zeros to demarcarte start of an iteration cycle
feature_vector = (0, 0, 0, 0, 0, 0, [0, 0, 0, 0], 0, 0, 0, 0, 0)
training_data.append(feature_vector)

INITIALIZE MODEL

In [10]:
import yaml
import os
import openai as OpenAI
import openai
import json


# Load the YAML file
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

# Get the API key from the YAML file
api_key = config['openai']['api_key']

from pathlib import Path
import time
    
# Initialize the OpenAI client
client = openai.OpenAI(api_key=api_key)

#initialize request parameters
n_models = 2
system_task = "Invent rules for a board game with a scoring system on a 1-step scale and play that game until you reach score 5. Respond with 'FINISH' once a player has reached score 5. Do not ever mention the word 'FINISH' unless a player has reached 5 points."#
#"Order a Pizza near the Giesing Brewery in Munich. Deliver it to the following address: Arcisstraße 21, 80333 Munich, Germany. Payment with be handled by a human upon arrival of the pizza. Respond with 'FINISH' when you are done."task_step = system_task
num_step = 0
response = "Null"
task_step=system_task

orep.openai_request(client, n_models, num_step, system_task, task_step, response, response_outputs)


{
  "steps": [
    "- Define the objective of the game.",
    "- Determine the number of players.",
    "- Create the game board layout.",
    "- Establish the rules for player movement.",
    "- Define how players can score points.",
    "- Set up any special rules or conditions.",
    "- Play the game according to the rules until a player reaches 5 points."
  ]
}
{
  "tasks": [
    "- Define the objective of the game.",
    "- Determine the number of players.",
    "- Create the game board layout.",
    "- Establish the rules for player movement.",
    "- Define how players can score points.",
    "- Set up any special conditions or events.",
    "- Outline the winning conditions.",
    "- Play the game step-by-step until a player reaches 5 points."
  ]
}


GENERATE TRAINING PARAMETERS

In [11]:
fitness_scores = []

for response in response_outputs:
    response = response[num_step]
    text = list(json.loads(response['message']).values())[0]
    ag_rationality, model_type_label, agent_type_label = orep.evaluate_fitness(response["model_type"], response["agent_type"], problem_type)
    fitness_scores.append(ag_rationality)

# Calculate consensus score
consensus_score = orep.calculate_consensus_score(response_outputs)

# Calculate the action maximization score for each response
act_max_scores = []
for fitness in fitness_scores:
    ag_rationality = fitness
    act_max_score = orep.calculate_act_max_score(ag_rationality, consensus_score)
    act_max_scores.append(act_max_score)


/Users/Lukas_1/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [12]:

# # Output the scores
# for i, response in enumerate(responses):
#     print(f"Response from {response['model_type']} - Agent Rationality: {fitness_scores[i][0]}, Act Max Score: {act_max_scores[i]}")

# Select the best response
best_response_index = act_max_scores.index(max(act_max_scores))
best_response = response_outputs[best_response_index]
best_response_backup.append(best_response)
model_type = best_response[0]["model_type"]
act_max_score = max(act_max_scores)

messages.append(best_response)

In [13]:
error_rate = orep.evaluate_error_rate(messages[0][num_step]['message'], best_response_backup)
best_response_body = list(json.loads(best_response[num_step]['message']).values())[0]


In [14]:
problem_scope_label, problem_type_label, num_step, num_agents, num_steps_proposed, ratio_steps_left = orep.evaluate_environment_complexity(problem_scope, problem_type, num_step, n_models, best_response_body, num_steps_proposed)

In [15]:
#generate input vector

#orep.generate_feature_vector(act_max_score, model_type, problem_scope, problem_type, num_step, n_models, best_response_body, adaptation_rate, agent_type='LLM')

feature_vector = act_max_score, ag_rationality, model_type_label, agent_type_label, adaptation_rate, problem_scope_label, problem_type_label, num_step, n_models, num_steps_proposed, ratio_steps_left, error_rate

training_data.append(feature_vector)


In [18]:
best_response_body

['- Define the objective of the game.',
 '- Determine the number of players.',
 '- Create the game board layout.',
 '- Establish the rules for player movement.',
 '- Define how players can score points.',
 '- Set up the initial game state.',
 '- Play the game, taking turns according to the rules.',
 '- Track the score of each player.',
 '- Continue playing until a player reaches 5 points.',
 '- Declare the winner once a player reaches 5 points.']

UPDATE REQUEST PARAMETERS

In [16]:
task_step = best_response_body[num_step]
num_steps_proposed = len(best_response_body)
num_step =+ 1
response_outputs_backup.append(response_outputs)
response_outputs = []

orep.openai_request(client, n_models, num_step, system_task, task_step, response, response_outputs)

{
  "message": {
    "steps": [
      "- Define the objective of the game.",
      "- Determine the number of players.",
      "- Create the game board layout.",
      "- Establish the rules for player movement.",
      "- Define how players can score points.",
      "- Set up any special conditions or events.",
      "- Outline the winning conditions.",
      "- Play the game step-by-step until a player reaches 5 points."
    ],
    "solution": {
      "objective": "The objective of the game is to be the first player to reach 5 points by strategically moving across the game board and overcoming various challenges."
    }
  }
}
{
  "message": {
    "steps": [
      "- Define the objective of the game.",
      "- Determine the number of players.",
      "- Create the game board layout.",
      "- Establish the rules for player movement.",
      "- Define how players can score points.",
      "- Set up any special conditions or events.",
      "- Outline the winning conditions.",
      "

####
####INITIALIZE SUBSEQUENT RUNS
####

In [21]:
import json
import pandas as pd

def auto_generate_feature_vector(response_outputs, num_step, problem_scope, problem_type, n_models, adaptation_rate, num_steps_proposed, response_outputs_backup):
    """
    Generate a feature vector based on the responses and environment complexity.

    Args:
    - response_outputs: List of responses from the models.
    - num_step: Current step number.
    - problem_scope: Scope of the problem.
    - problem_type: Type of the problem.
    - n_models: Number of models.
    - adaptation_rate: Rate of adaptation (default is 0).

    Returns:
    - feature_vector: A tuple containing the feature vector.
    """
    # Initialize lists to store scores
    fitness_scores = []
    act_max_scores = []

    # Evaluate fitness for each response
    for response in response_outputs:
        response = response[num_step]
        #text = list(json.loads(response['message']).values())[0]
        ag_rationality, model_type_label, agent_type_label = orep.evaluate_fitness(response["model_type"], response["agent_type"], problem_type)
        fitness_scores.append(ag_rationality)

    # Calculate consensus score
    consensus_score = orep.calculate_consensus_score(response_outputs)

    # Calculate action maximization scores
    for fitness in fitness_scores:
        ag_rationality = fitness
        act_max_score = orep.calculate_act_max_score(ag_rationality, consensus_score)
        act_max_scores.append(act_max_score)

    # Select the best response
    best_response_index = act_max_scores.index(max(act_max_scores))
    best_response = response_outputs[best_response_index]
    best_response_backup.append(best_response)
    model_type = best_response[num_step]["model_type"]
    model_type_label = {"gpt-3.5": 1, "gpt-4": 2, "gpt-4o": 2, "gemini": 3, "gemini-ultra": 4, "mistral": 5}[model_type] #convert model type to label

    act_max_score = max(act_max_scores)

    messages = [best_response]
    #ro_df_columns = ["model_1", "model_2"]
    if num_step > 1:
        last_responses = [response_outputs_backup[num_step], response_outputs_backup[num_step-1]]
        error_rate = orep.evaluate_error_rate(messages[0][num_step]['message'], best_response_backup, responses=last_responses, num_step=num_step)
    else:
        error_rate = orep.evaluate_error_rate(messages[0][num_step]['message'], best_response_backup, responses=None, num_step=num_step)

    best_response_body = list(json.loads(best_response[num_step]['message']).values())[0]
    print(best_response_body, error_rate)

    problem_scope_label, problem_type_label, num_step, n_models, num_steps_proposed, ratio_steps_left = orep.evaluate_environment_complexity(
        problem_scope, problem_type, num_step, n_models, best_response_body, num_steps_proposed)

    # Generate the feature vector
    feature_vector = (
        act_max_score,
        ag_rationality,
        model_type_label,
        agent_type_label,
        adaptation_rate,
        problem_scope_label,
        problem_type_label,
        num_step,
        n_models,
        num_steps_proposed,
        ratio_steps_left,
        error_rate
    )

    return feature_vector, best_response_body

In [18]:
def contains_finish(response_outputs_backup):
    for response in response_outputs_backup:
        if isinstance(response, list):
            for sub_response in response:
                for key, value in sub_response.items():
                    if 'message' in value:
                        message = value['message']
                        if isinstance(message, dict):
                            if any("FINISH" in msg for msg in message.values()):
                                return True
                        elif isinstance(message, str):
                            if "FINISH" in message:
                                return True
        elif isinstance(response, dict):
            for key, value in response.items():
                if 'message' in value:
                    message = value['message']
                    if isinstance(message, dict):
                        if any("FINISH" in msg for msg in message.values()):
                            return True
                    elif isinstance(message, str):
                        if "FINISH" in message:
                            return True
    return False

In [22]:
response_outputs_backup

[[{0: {'model_type': 'gpt-4o',
    'agent_type': 'LLM',
    'message': '{\n  "steps": [\n    "- Define the objective of the game.",\n    "- Determine the number of players.",\n    "- Create the game board layout.",\n    "- Establish the rules for player movement.",\n    "- Define how players can score points.",\n    "- Set up the initial game state.",\n    "- Play the game, taking turns according to the rules.",\n    "- Track the score of each player.",\n    "- Continue playing until a player reaches 5 points.",\n    "- Declare the winner once a player reaches 5 points."\n  ]\n}'}},
  {0: {'model_type': 'gpt-4o',
    'agent_type': 'LLM',
    'message': '{\n  "tasks": [\n    "- Define the objective of the game.",\n    "- Determine the number of players.",\n    "- Create the game board layout.",\n    "- Establish the rules for player movement.",\n    "- Develop a scoring system on a 1-step scale.",\n    "- Set the conditions for winning the game.",\n    "- Play the game according to the 

In [19]:
def process_steps(response_outputs, best_response_body, num_step, problem_scope, problem_type, n_models, adaptation_rate, num_steps_proposed, response_outputs_backup, sub_layer=0, finished=False):
    bullet_markers = ['-', '*', '•']
    finished = finished
    i = len(best_response_body) * 2
    j = 0

    while not finished:
        
        feature_vector, response = auto_generate_feature_vector(response_outputs, num_step, problem_scope, problem_type, n_models, adaptation_rate, num_steps_proposed, response_outputs_backup)
        training_data.append(feature_vector)
        
        task_step = best_response_body[num_step]
        if num_step < (len(best_response_body) - 1):
            num_step += 1
        
        response_outputs_backup.append(response_outputs)
        response_outputs = []
        sub_layer = 0

        orep.openai_request(client, n_models, num_step, system_task, task_step, response, response_outputs)

        if any("FINISH" in output for output in response_outputs):
            return True, sub_layer
        #THIS NEEDS A GLOBAL VARIABLE THAT IS EITHER 1 OR 0 AND PROHIBITS THE ESTABLISHMENT OF DEEPER LOOP LAYERS IF 1.
        # If the response contains sub-steps, process them recursively
        for response in response_outputs:
            response_body = json.loads(response[num_step]['message'])

            if 'message' not in response_body:
                response_body['message'] = str(response_body)
            
            if 'steps:' in response_body['message']:
                sub_steps = []
                for line in response_body['message'].split('\n'):
                    if any(line.strip().startswith(marker) for marker in bullet_markers):
                        sub_steps.append(line.strip())
                
                if sub_steps and sub_layer == 0:
                    sub_layer += 1
                    sub_response_body = ", ".join(sub_steps)
                    for step in sub_steps:
                        finished, sub_layer = process_steps(response_outputs, sub_response_body, num_step, problem_scope, problem_type, n_models, adaptation_rate, num_steps_proposed, response_output_backup, sub_layer)
                        return finished, sub_layer
                    
            elif 'steps:' in response_body:
                sub_steps = []
                for line in response_body['message'].split('\n'):
                    if any(line.strip().startswith(marker) for marker in bullet_markers):
                        sub_steps.append(line.strip())
                
                if sub_steps and sub_layer == 0:
                    sub_layer += 1
                    sub_response_body = ", ".join(sub_steps)
                    for step in sub_steps:
                        finished, sub_layer = process_steps(response_outputs, sub_response_body, num_step, problem_scope, problem_type, n_models, adaptation_rate, num_steps_proposed, response_output_backup, sub_layer)
                        return finished, sub_layer
                    
        print("updated sub_layer equals:" + str(sub_layer))
        j += 1
        if j == i:
            break
    return False, sub_layer


In [35]:
responses = [response_outputs_backup[num_step][0], response_outputs_backup[num_step-1][0]]
responses

[{1: {'model_type': 'gpt-4o',
   'agent_type': 'LLM',
   'message': '{\n  "message": {\n    "steps": [\n      "- Define the objective of the game.",\n      "- Determine the number of players.",\n      "- Create the game board layout.",\n      "- Establish the rules for player movement.",\n      "- Define how players can score points.",\n      "- Set up any special conditions or events.",\n      "- Outline the winning conditions.",\n      "- Play the game step-by-step until a player reaches 5 points."\n    ],\n    "solution": {\n      "objective": "The objective of the game is to be the first player to reach 5 points by strategically moving across the game board and overcoming various challenges."\n    }\n  }\n}'}},
 {0: {'model_type': 'gpt-4o',
   'agent_type': 'LLM',
   'message': '{\n  "steps": [\n    "- Define the objective of the game.",\n    "- Determine the number of players.",\n    "- Create the game board layout.",\n    "- Establish the rules for player movement.",\n    "- Defi

In [38]:
parsed_responses = []
for response in responses:
    if isinstance(response, dict):
        for key in response:
            if isinstance(response[key], dict) and 'message' in response[key]:
                parsed_message = json.loads(response[key]['message'])
                parsed_responses.append(parsed_message)
    elif isinstance(response, dict) and 'message' in response:
        parsed_message = json.loads(response['message'])
        parsed_responses.append(parsed_message)
print("parsed_responses: ")
print(parsed_responses)

parsed_responses: 
[{'message': {'steps': ['- Define the objective of the game.', '- Determine the number of players.', '- Create the game board layout.', '- Establish the rules for player movement.', '- Define how players can score points.', '- Set up any special conditions or events.', '- Outline the winning conditions.', '- Play the game step-by-step until a player reaches 5 points.'], 'solution': {'objective': 'The objective of the game is to be the first player to reach 5 points by strategically moving across the game board and overcoming various challenges.'}}}, {'steps': ['- Define the objective of the game.', '- Determine the number of players.', '- Create the game board layout.', '- Establish the rules for player movement.', '- Define how players can score points.', '- Set up any special rules or conditions.', '- Play the game according to the rules until a player reaches 5 points.']}]


In [39]:
importlib.reload(orep)

<module 'oracle_repo' from '/Users/Lukas_1/Library/Mobile Documents/com~apple~CloudDocs/Repos/Oracle/oracle_repo.py'>

In [114]:
# [list(json.loads((response[num_step][next(iter(response[i]))]['message'])).values()) for response in responses for i in range((len(responses)))]

[[{'steps': ['- Define the objective of the game.',
    '- Determine the number of players.',
    '- Create the game board layout.',
    '- Establish the rules for player movement.',
    '- Develop a scoring system on a 1-step scale.',
    '- Set the conditions for winning the game.',
    '- Play the game according to the rules until a player reaches a score of 5.'],
   'solution': {'objective': 'The objective of the game is to navigate the game board, collect points, and be the first player to reach a score of 5.'}}],
 [{'steps': ['- Define the objective of the game.',
    '- Determine the number of players.',
    '- Create the game board layout.',
    '- Establish the rules for player movement.',
    '- Develop a scoring system on a 1-step scale.',
    '- Set the conditions for winning the game.',
    '- Play the game according to the rules until a player reaches a score of 5.'],
   'solution': {'objective': 'The objective of the game is to navigate the game board, collect points, an

In [130]:
# responses = [response_outputs_backup[num_step][0], response_outputs_backup[num_step-1][0]]
# responses
# #[list(json.loads((response[num_step][next(iter(response[i]))]['message'])).values()) for response in responses for i in range(len(response))]
# orep.evaluate_error_rate(messages[0][0]['message'], best_response_backup, responses=responses, num_step=num_step)

0.0

In [98]:
# [list(json.loads((response[num_step][next(iter(response[i]))]['message'])).values()) for response in responses for i in range(len(response))]

[[{'steps': ['- Define the objective of the game.',
    '- Determine the number of players.',
    '- Create the game board layout.',
    '- Establish the rules for player movement.',
    '- Develop a scoring system on a 1-step scale.',
    '- Set the conditions for winning the game.',
    '- Play the game according to the rules until a player reaches a score of 5.'],
   'solution': {'objective': 'The objective of the game is to navigate the game board, collect points, and be the first player to reach a score of 5.'}}],
 [{'steps': ['- Define the objective of the game.',
    '- Determine the number of players.',
    '- Create the game board layout.',
    '- Establish the rules for player movement.',
    '- Develop a scoring system on a 1-step scale.',
    '- Set the conditions for winning the game.',
    '- Play the game according to the rules until a player reaches a score of 5.'],
   'solution': {'objective': 'The objective of the game is to navigate the game board, collect points, an

In [2]:
    responses = [
        json.loads(responses[i][key]['message'])
        for i in range(len(responses))
        for key in responses[i]
    ]
    ##--> need to test this object, as consensus score and error rate are called twice in the script for different dict object structures...

NameError: name 'responses' is not defined

In [47]:
finished = False
sub_layer = 0

finished, sub_layer = process_steps(response_outputs, best_response_body, num_step, problem_scope, problem_type, n_models, adaptation_rate, num_steps_proposed, response_outputs_backup, sub_layer)


response: 
{1: {'model_type': 'gpt-4o', 'agent_type': 'LLM', 'message': '{\n  "message": {\n    "steps": [\n      "- Define the objective of the game.",\n      "- Determine the number of players.",\n      "- Create the game board layout.",\n      "- Establish the rules for player movement.",\n      "- Define how players can score points.",\n      "- Set up any special conditions or events.",\n      "- Outline the winning conditions.",\n      "- Play the game step-by-step until a player reaches 5 points."\n    ],\n    "solution": {\n      "objective": "The objective of the game is to be the first player to reach 5 points by strategically moving across the game board and overcoming various challenges."\n    }\n  }\n}'}}
parsed_responses: 
[{'message': {'steps': ['- Define the objective of the game.', '- Determine the number of players.', '- Create the game board layout.', '- Establish the rules for player movement.', '- Define how players can score points.', '- Set up any special conditi

Traceback (most recent call last):
  File "/Users/Lukas_1/Library/Mobile Documents/com~apple~CloudDocs/Repos/Oracle/oracle_repo.py", line 361, in openai_request
    response = client.chat.completions.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/Lukas_1/anaconda3/lib/python3.11/site-packages/openai/_utils/_utils.py", line 303, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/Users/Lukas_1/anaconda3/lib/python3.11/site-packages/openai/resources/chat/completions.py", line 598, in create
    return self._post(
           ^^^^^^^^^^^
  File "/Users/Lukas_1/anaconda3/lib/python3.11/site-packages/openai/_base_client.py", line 1086, in post
    return cast(ResponseT, self.request(cast_to, opts, stream=stream, stream_cls=stream_cls))
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/Lukas_1/anaconda3/lib/python3.11/site-packages/openai/_base_client.py", line 846, in request
  

In [46]:
importlib.reload(orep)

<module 'oracle_repo' from '/Users/Lukas_1/Library/Mobile Documents/com~apple~CloudDocs/Repos/Oracle/oracle_repo.py'>

In [23]:
t_df_columns = ["act_max_score", "ag_rationality", "model_type_label", "agent_type_label", "adaptation_rate", "problem_scope_label", "problem_type_label", "num_step", "n_models", "num_steps_proposed", "ratio_steps_left", "error_rate"]
training_data_df = pd.DataFrame(training_data, columns=t_df_columns)
training_data_df

,act_max_score,ag_rationality,model_type_label,agent_type_label,adaptation_rate,problem_scope_label,problem_type_label,num_step,n_models,num_steps_proposed,ratio_steps_left,error_rate
0,0.000000,0.000,0,0,0,0,"[0, 0, 0, 0]",0,0,0,0.000000,0.0
1,32.633882,108.696,2,1,0,1,"[1, 0, 0, 0]",0,2,7,1.000000,0.0
2,32.908800,108.696,2,1,0,1,"[1, 0, 0, 0]",1,2,7,0.857143,0.0
3,32.908800,108.696,2,1,0,1,"[1, 0, 0, 0]",1,2,7,0.857143,0.0
4,32.908800,108.696,2,1,0,1,"[1, 0, 0, 0]",2,2,7,0.714286,0.0
5,32.908800,108.696,2,1,0,1,"[1, 0, 0, 0]",3,2,7,0.571429,0.0
6,32.908800,108.696,2,1,0,1,"[1, 0, 0, 0]",4,2,7,0.428571,0.0
7,32.908800,108.696,2,1,0,1,"[1, 0, 0, 0]",5,2,7,0.285714,0.0
8,32.908800,108.696,2,1,0,1,"[1, 0, 0, 0]",6,2,7,0.142857,0.0
9,32.908800,108.696,2,1,0,1,"[1, 0, 0, 0]",6,2,7,0.142857,0.0


In [27]:
pd.DataFrame(best_response_backup)

,0,1,2,3,4,5,6
0,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ...",NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ...",NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ...",NaN,NaN,NaN,NaN
3,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ...",NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ...",NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ...",NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ...",NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ...",NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ..."
9,NaN,NaN,NaN,NaN,NaN,NaN,"{'model_type': 'gpt-4o', 'agent_type': 'LLM', ..."


In [761]:
import pandas as pd
t_df_columns = ["act_max_score", "ag_rationality", "model_type_label", "agent_type_label", "adaptation_rate", "problem_scope_label", "problem_type_label", "num_step", "n_models", "num_steps_proposed", "ratio_steps_left", "error_rate"]
ro_df_columns = ["model_1", "model_2"]
training_data_df = pd.DataFrame(training_data, columns=t_df_columns)
response_outputs_backup_df = pd.DataFrame(response_outputs_backup, columns=ro_df_columns)
best_response_backup_df = pd.DataFrame(best_response_backup, columns=["best_response_backup"])
training_data_df.to_csv("training_data.csv", index=False)
response_outputs_backup_df.to_csv("response_outputs_backup.csv", index=False)

In [28]:
df = pd.DataFrame(best_response_backup)

def consolidate_non_nan_entries(df):
    consolidated_list = []
    for index, row in df.iterrows():
        consolidated_row = {}
        for item in row.dropna():
            consolidated_row.update(item)
        consolidated_list.append(consolidated_row)
    return pd.DataFrame(consolidated_list)

# Consolidate the DataFrame
consolidated_df = consolidate_non_nan_entries(df)
consolidated_df
#consolidated_df.to_csv("best_response_backup.csv", index=False)

,model_type,agent_type,message
0,gpt-4o,LLM,"{\n ""steps"": [\n ""- Define the objective o..."
1,gpt-4o,LLM,"{\n ""message"": {\n ""steps"": [\n ""- De..."
2,gpt-4o,LLM,"{\n ""steps"": [\n ""- Define the objective o..."
3,gpt-4o,LLM,"{\n ""message"": {\n ""steps"": [\n ""- De..."
4,gpt-4o,LLM,"{\n ""message"": {\n ""steps"": [\n ""- De..."
5,gpt-4o,LLM,"{\n ""message"": {\n ""steps"": [\n ""- De..."
6,gpt-4o,LLM,"{\n ""message"": {\n ""steps"": [\n ""- De..."
7,gpt-4o,LLM,"{\n ""message"": {\n ""steps"": [\n ""- De..."
8,gpt-4o,LLM,"{\n ""steps"": [\n ""- Define the objective o..."
9,gpt-4o,LLM,"{\n ""message"": {\n ""steps"": [\n ""- De..."


In [30]:
#response_outputs_backup_df["model_1"][1][1]['message']
#-> use this code to calculate the error rate by comparing the similarities between word vectors. If similaritiy is > 0.95 for t and t+1 then add +1 to error rate and +1 to strike_counter. It strike_counter hits 5, break the loop (possibly ask the model to consider if it is finished).
response_outputs_backup

[[{0: {'model_type': 'gpt-4o',
    'agent_type': 'LLM',
    'message': '{\n  "steps": [\n    "- Define the objective of the game.",\n    "- Determine the number of players.",\n    "- Create the game board layout.",\n    "- Establish the rules for player movement.",\n    "- Define how players can score points.",\n    "- Set up any special rules or conditions.",\n    "- Play the game according to the rules until a player reaches 5 points."\n  ]\n}'}},
  {0: {'model_type': 'gpt-4o',
    'agent_type': 'LLM',
    'message': '{\n  "tasks": [\n    "- Define the objective of the game.",\n    "- Determine the number of players.",\n    "- Create the game board layout.",\n    "- Establish the rules for player movement.",\n    "- Define how players can score points.",\n    "- Set up any special conditions or events.",\n    "- Outline the winning conditions.",\n    "- Play the game step-by-step until a player reaches 5 points."\n  ]\n}'}}],
 [{1: {'model_type': 'gpt-4o',
    'agent_type': 'LLM',
  

In [ ]:
import pandas as pd
t_df_columns = ["act_max_score", "ag_rationality", "model_type_label", "agent_type_label", "adaptation_rate", "problem_scope_label", "problem_type_label", "num_step", "n_models", "num_steps_proposed", "ratio_steps_left", "error_rate"]
ro_df_columns = ["model_1", "model_2"]
training_data_df = pd.DataFrame(training_data, columns=t_df_columns)
response_outputs_backup_df = pd.DataFrame(response_outputs_backup, columns=ro_df_columns)
best_response_backup_df = pd.DataFrame(best_response_backup, columns=["best_response_backup"])
training_data_df.to_csv("training_data.csv", index=False)
response_outputs_backup_df.to_csv("response_outputs_backup.csv", index=False)

In [774]:
for item in best_response_backup:
    print(item)

{0: {'model_type': 'gpt-4o', 'agent_type': 'LLM', 'message': '{\n  "steps": [\n    "- Define the objective of the game.",\n    "- Determine the number of players.",\n    "- Establish the game board layout.",\n    "- Create the rules for player movement.",\n    "- Develop the scoring system.",\n    "- Outline the win conditions.",\n    "- Play the game until a player reaches 5 points."\n  ]\n}'}}
{1: {'model_type': 'gpt-4o', 'agent_type': 'LLM', 'message': '{\n  "message": {\n    "steps": [\n      "- Define the objective of the game",\n      "- Establish the rules for player actions",\n      "- Create a scoring system",\n      "- Determine the conditions for winning",\n      "- Play the game according to the rules",\n      "- Track the scores of each player",\n      "- Continue playing until a player reaches 5 points"\n    ],\n    "solution": {\n      "objective": "The objective of the game is to be the first player to reach 5 points by following the established rules and scoring system

In [771]:
# Function to flatten the dictionary
def flatten_dict(d, parent_key='', sep='_'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            for i, item in enumerate(v):
                items.extend(flatten_dict({f"{new_key}{sep}{i}": item}).items())
        else:
            items.append((new_key, v))
    return dict(items)

# Flatten the dictionary
flattened_data = flatten_dict(best_response_backup)

best_response_backup_df = pd.DataFrame([flattened_data])

del(flattened_data)

AttributeError: 'list' object has no attribute 'items'

In [764]:
best_response_backup_df = pd.DataFrame(best_response_backup, columns=["best_response_backup"])
best_response_backup_df

,best_response_backup
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
6,NaN


DEFINE THE NETWORK

In [ ]:
# Assume generate_feature_vector function is defined as in previous messages

class LSTMNetwork(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(LSTMNetwork, self).__init__()
        self.hidden_size = hidden_size
        
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(1, x.size(0), self.hidden_size).to(x.device)
        
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]  # Get the last time step output
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

class FeatureDataset(Dataset):
    def __init__(self, feature_vectors, labels):
        self.feature_vectors = feature_vectors
        self.labels = labels
    
    def __len__(self):
        return len(self.feature_vectors)
    
    def __getitem__(self, idx):
        return torch.tensor(self.feature_vectors[idx], dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.float32)

# Example feature vectors and labels
feature_vectors = [orep.generate_feature_vector(env_params, responses, api_response_text, rationality_params, accuracy, benchmark_performance_score, messages) for _ in range(100)]
labels = np.random.randint(0, 2, size=100)  # Binary labels: 1 for success, 0 for failure

# Create dataset and dataloader
dataset = FeatureDataset(feature_vectors, labels)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

# Hyperparameters
input_size = len(feature_vectors[0])  # Number of features in the feature vector
hidden_size = 15

# Initialize the network, loss function, and optimizer
model = LSTMNetwork(input_size, hidden_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

TRAIN THE NETWORK

In [ ]:
# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    for i, (features, labels) in enumerate(dataloader):
        # Reshape input to (batch_size, seq_length, input_size)
        features = features.unsqueeze(1)
        
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs.squeeze(), labels)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

print('Training completed.')

PREDICT AND EVALUATE SUCCESS

In [ ]:
def predict_success(model, feature_vector, threshold=0.55):
    model.eval()
    feature_tensor = torch.tensor(feature_vector, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # Shape (1, 1, input_size)
    with torch.no_grad():
        probability = model(feature_tensor).item()
    return probability, probability > threshold

# Example usage
test_feature_vector = generate_feature_vector(env_params, responses, api_response_text, rationality_params, accuracy, benchmark_performance_score, messages)
probability, is_success = predict_success(model, test_feature_vector)
print(f'Probability of Success: {probability:.4f}, Success: {is_success}')